In [2]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from src.target import LINES, load_policies, load_claims, match_claims

pol = {l: load_policies(l, "train") for l in LINES}
claims = load_claims()
m = pol["motor"]
mc = match_claims(claims, m, "motor")

# [1] 연도별 계약 구성
flags = ["driver_assist", "telematics_opt_in", "new_business", "named_young_driver"]
print("[1a] share by year")
print(m.groupby("year")[flags].mean().round(3))
print("\n[1b] averages by year")
print(m.groupby("year")[["driver_age", "vehicle_age", "annual_mileage", "ncb_years"]].mean().round(2))
print("\n[1c] cover_type mix by year")
print(m.groupby("year").cover_type.value_counts(normalize=True).unstack().round(3))

# [2] 집단별 빈도 (own_damage, third_party_prop, 1,000 노출당)
cnt = mc.groupby(["row_id", "peril"]).size().unstack(fill_value=0)
m2 = (m[["row_id", "year", "exposure_clean", "driver_assist", "telematics_opt_in"]]
      .merge(cnt[["own_damage", "third_party_prop"]], left_on="row_id", right_index=True, how="left")
      .fillna({"own_damage": 0, "third_party_prop": 0}))

for var in ["driver_assist", "telematics_opt_in"]:
    g = m2.groupby(["year", var])[["own_damage", "third_party_prop", "exposure_clean"]].sum()
    g["od_per_1000"] = g.own_damage / g.exposure_clean * 1000
    g["tpp_per_1000"] = g.third_party_prop / g.exposure_clean * 1000
    print(f"\n[2] frequency by year and {var}")
    print(g[["exposure_clean", "od_per_1000", "tpp_per_1000"]].round(1).unstack(var))

[1a] share by year
      driver_assist  telematics_opt_in  new_business  named_young_driver
year                                                                    
2021          0.301              0.117         0.120               0.055
2022          0.301              0.122         0.152               0.054
2023          0.301              0.127         0.153               0.053
2024          0.302              0.130         0.154               0.052

[1b] averages by year
      driver_age  vehicle_age  annual_mileage  ncb_years
year                                                    
2021       50.80         6.57        11333.72       6.61
2022       50.59         7.46        11344.95       7.08
2023       50.39         8.20        11356.64       7.44
2024       50.26         8.79        11362.86       7.71

[1c] cover_type mix by year
cover_type  comprehensive  third_party_fire_theft  third_party_only
year                                                               
2021         

In [3]:
from src.target import build_model_target

mt, comp = build_model_target(claims, pol["motor"])
print("[completeness by policy year]")
print(comp.round(3))

# 검증: 모든 연도가 2025년 수준으로 맞춰졌다면 비슷한 값이 나와야 함
check = mt.groupby("year")[["target", "exposure_clean", "weight"]].sum()
check["raw_per_exp"] = check.target / check.exposure_clean
check["adj_per_weight"] = check.target / check.weight
print("\n[2025-level cost per exposure by year]")
print(check[["raw_per_exp", "adj_per_weight"]].round(1))

[completeness by policy year]
2021    1.000
2022    0.997
2023    0.979
2024    0.816
Name: completeness, dtype: float64

[2025-level cost per exposure by year]
      raw_per_exp  adj_per_weight
year                             
2021        550.7           550.8
2022        547.0           548.4
2023        539.2           551.0
2024        440.6           540.0
